# ДЗ 2.1. Сборка таблицы SCD2

Таблица собирается заранее скриптом `python3 build_scd2.py` (в терминале Jupyter) и лежит в S3: `s3a://hw2-1/users_scd2/`.

In [2]:
!python3 build_scd2.py

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 19:01:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/04 19:01:43 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
Готово: 379 строк в s3a://hw2-1/users_scd2/                                     


In [3]:
from build_scd2 import TARGET, create_spark

spark = create_spark()

(spark.read.parquet(TARGET) # читает Parquet-файлы из S3 как таблицу (DataFrame)
 .createOrReplaceTempView("users_scd2") # регистрируем её под именем users_scd2 для SQL-запросов
 )

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/04 19:03:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/04 19:03:53 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


## Задание 4. Количество строк в полученной SCD2 таблице

In [4]:
COUNT_ROWS_SQL = "SELECT COUNT(*) AS scd2_rows FROM users_scd2"

In [5]:
spark.sql(COUNT_ROWS_SQL).show()

+---------+
|scd2_rows|
+---------+
|      379|
+---------+



## Задание 5. Активный user_login пользователя X на момент Y

Версия активна в момент Y, если `valid_from <= Y < valid_to`.

In [8]:
# :user_id и :at_time — параметры запроса (X и Y), значения передаются через args.
ACTUAL_LOGIN_SQL = """
SELECT user_login
FROM users_scd2
WHERE user_id = :user_id
  AND valid_from <= CAST(:at_time AS TIMESTAMP)
  AND (valid_to IS NULL OR CAST(:at_time AS TIMESTAMP) < valid_to)
"""

Пример для пользователя user_id = 47

In [9]:
spark.sql("SELECT * FROM users_scd2 WHERE user_id = 47 ORDER BY valid_from").show(truncate=False)

# Логин пользователя 15 в разные моменты Y. До первого события результат пустой.
for y in ["2026-09-01 00:03:53", "2026-09-01 00:03:54", "2026-09-01 01:24:15", "2026-09-01 01:24:16", "2100-01-01 00:00:00"]:
    rows = spark.sql(
        ACTUAL_LOGIN_SQL,
        args={"user_id": 47, "at_time": y}).collect()

    print(f"X=47, Y={y} -> {[row.user_login for row in rows]}")

+-------+---------------+-------------------------------+------------+-------------------+-------------------+----------+
|user_id|user_fio       |user_email                     |user_login  |valid_from         |valid_to           |is_current|
+-------+---------------+-------------------------------+------------+-------------------+-------------------+----------+
|47     |Allison Collins|allison.collins.47@mail.example|acollins47  |2026-09-01 00:00:44|2026-09-01 02:47:46|false     |
|47     |Allison Collins|allison.collins.47@mail.example|acollins47r1|2026-09-01 02:47:46|NULL               |true      |
+-------+---------------+-------------------------------+------------+-------------------+-------------------+----------+

X=47, Y=2026-09-01 00:03:53 -> ['acollins47']
X=47, Y=2026-09-01 00:03:54 -> ['acollins47']
X=47, Y=2026-09-01 01:24:15 -> ['acollins47']
X=47, Y=2026-09-01 01:24:16 -> ['acollins47']
X=47, Y=2100-01-01 00:00:00 -> ['acollins47r1']


## Проверка: у каждого пользователя ровно одна актуальная версия (ожидается 0)

In [10]:
spark.sql("""
    SELECT COUNT(*) AS users_with_wrong_current
    FROM (SELECT user_id FROM users_scd2 GROUP BY user_id HAVING SUM(CAST(is_current AS INT)) <> 1)
""").show()

+------------------------+
|users_with_wrong_current|
+------------------------+
|                       0|
+------------------------+



In [11]:
# Освобождение ядер кластера
spark.stop()